In [1]:
# !pip install googlesearch-python beautifulsoup4 requests python-dotenv

In [2]:
# Cell 2: Import libraries
import os
import json
import re
from typing import TypedDict, List, Dict, Any, Annotated
from datetime import datetime
import requests
from bs4 import BeautifulSoup
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
import operator
from dotenv import load_dotenv

load_dotenv()

# Load and verify API keys from environment
def load_api_keys():
    google_key = os.getenv("GOOGLE_API_KEY")
    tavily_key = os.getenv("TAVILY_API_KEY")
    
    if not google_key:
        raise ValueError("GOOGLE_API_KEY not found in environment variables")
    if not tavily_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables")
    
    return google_key, tavily_key

# Load keys
GOOGLE_API_KEY, TAVILY_API_KEY = load_api_keys()
print("✓ API keys loaded successfully from environment")

c:\Users\Admin\Desktop\School_Projects\git repositories\SPARK-v2\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✓ API keys loaded successfully from environment


In [3]:
# Cell 3: Setup LLM
_llm = None

def get_llm():
    global _llm
    if _llm is None:
        _llm = ChatGoogleGenerativeAI(
            model="gemini-3.1-flash-lite-preview",
            temperature=0,
            google_api_key=GOOGLE_API_KEY,
            max_retries=5,
        )
    return _llm

In [4]:
# Cell 4: Define agent state
class JobAgentState(TypedDict):
    query: str
    search_keywords: str
    search_results: List[str]
    scraped_content: List[Dict[str, Any]]
    extracted_jobs: List[Dict[str, Any]]
    filtered_jobs: List[Dict[str, Any]]
    ranked_jobs: List[Dict[str, Any]]
    final_answer: str
    user_skills: List[str]
    user_resume: str

In [5]:
# Cell 5: FIXED - Handle content as list
def understand_query(state: JobAgentState) -> JobAgentState:
    """Convert user query into search keywords"""
    llm = get_llm()
    
    prompt = ChatPromptTemplate.from_template(
        """You are a job search expert. Convert this query into search keywords for Google.
        
Query: {query}

Return ONLY the search keywords. Add "Vietnam jobs" or "Vietnam careers" if location is Vietnam.
Examples:
- "Find Software Engineer jobs in Da Nang" -> "Software Engineer jobs Da Nang Vietnam"
- "What companies hire Python developers?" -> "Python developer jobs Vietnam companies hiring"

Keywords:"""
    )
    
    chain = prompt | llm
    result = chain.invoke({"query": state["query"]})
    
    # Fix: Extract text from content list
    if hasattr(result, 'content'):
        if isinstance(result.content, list):
            # Content is a list of dicts with 'text' key
            keywords = result.content[0]['text'].strip()
        else:
            keywords = result.content.strip()
    else:
        keywords = str(result).strip()
    
    # Remove quotes if present
    keywords = keywords.strip('"\'')
    
    print(f"Search keywords: {keywords}")
    state["search_keywords"] = keywords
    return state

In [6]:
# Cell 6: Node 2 - Search the web using Tavily
from tavily import TavilyClient

def search_web(state: JobAgentState) -> JobAgentState:
    """Search for job listings using Tavily"""
    keywords = state["search_keywords"]
    urls = []
    
    try:
        # Search using Tavily
        client = TavilyClient(api_key=TAVILY_API_KEY)
        results = client.search(
            query=keywords,
            max_results=10,
            topic="general",
            include_domains=None,
            exclude_domains=None
        )
        
        # Extract URLs from Tavily results
        if "results" in results:
            for item in results["results"]:
                urls.append(item["url"])
        
        print(f"Tavily found {len(urls)} URLs for: {keywords}")
    except Exception as e:
        print(f"Tavily search error: {e}")
    
    state["search_results"] = urls
    return state

In [7]:
# Cell 7: Node 3 - Scrape web pages
def scrape_content(state: JobAgentState) -> JobAgentState:
    """Scrape content from URLs"""
    scraped = []
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    for url in state["search_results"][:5]:  # Limit to 5 pages
        try:
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Remove script and style tags
            for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
                tag.decompose()
            
            text = soup.get_text(separator=' ', strip=True)
            # Limit text length
            text = text[:5000]
            
            scraped.append({
                "url": url,
                "content": text
            })
            print(f"Scraped: {url[:50]}...")
        except Exception as e:
            print(f"Failed to scrape {url}: {e}")
    
    state["scraped_content"] = scraped
    return state

In [8]:
# Cell 8: FIXED - Handle content as list
def extract_jobs(state: JobAgentState) -> JobAgentState:
    """Extract structured job data from scraped content"""
    llm = get_llm()
    all_jobs = []
    
    for page in state["scraped_content"]:
        prompt = ChatPromptTemplate.from_template(
            """Extract job information from this text. Return a JSON list of jobs.

Text: {content}

Return format:
[
  {{
    "title": "Job Title",
    "company": "Company Name",
    "location": "City, Country",
    "salary": "salary range or null",
    "requirements": ["requirement1", "requirement2"],
    "responsibilities": ["duty1", "duty2"],
    "skills": ["skill1", "skill2"],
    "years_of_experience": "X years or null",
    "seniority": "Junior/Mid/Senior or null",
    "employment_type": "Full-time/Part-time/Contract or null",
    "remote": true or false,
    "apply_url": "{url}"
  }}
]

If no jobs found, return [].
Return ONLY valid JSON, no other text."""
        )
        
        try:
            chain = prompt | llm
            result = chain.invoke({
                "content": page["content"][:3000],
                "url": page["url"]
            })
            
            # Fix: Extract text from content list
            if hasattr(result, 'content'):
                if isinstance(result.content, list):
                    response_text = result.content[0]['text'].strip()
                else:
                    response_text = result.content.strip()
            else:
                response_text = str(result).strip()
            
            # Remove markdown code blocks if present
            response_text = re.sub(r'^```json\s*', '', response_text)
            response_text = re.sub(r'\s*```$', '', response_text)
            
            jobs = json.loads(response_text)
            if isinstance(jobs, list):
                all_jobs.extend(jobs)
                print(f"Extracted {len(jobs)} jobs from {page['url'][:50]}...")
        except Exception as e:
            print(f"Extraction error: {e}")
    
    state["extracted_jobs"] = all_jobs
    print(f"Total extracted jobs: {len(all_jobs)}")
    return state

In [9]:
# Cell 9: FIXED - Handle content as list
def filter_jobs(state: JobAgentState) -> JobAgentState:
    """Filter jobs based on original query"""
    llm = get_llm()
    
    if not state["extracted_jobs"]:
        state["filtered_jobs"] = []
        return state
    
    prompt = ChatPromptTemplate.from_template(
        """Filter these jobs based on the user query. Keep only relevant jobs.

Query: {query}
Jobs: {jobs}

Return a JSON list of job indices to keep (0-indexed).
Example: [0, 2, 5, 7]

If all jobs are relevant, return all indices.
Return ONLY the JSON list, no other text."""
    )
    
    try:
        chain = prompt | llm
        result = chain.invoke({
            "query": state["query"],
            "jobs": json.dumps(state["extracted_jobs"][:20], ensure_ascii=False)
        })
        
        # Fix: Extract text from content list
        if hasattr(result, 'content'):
            if isinstance(result.content, list):
                response_text = result.content[0]['text'].strip()
            else:
                response_text = result.content.strip()
        else:
            response_text = str(result).strip()
        
        response_text = re.sub(r'^```json\s*', '', response_text)
        response_text = re.sub(r'\s*```$', '', response_text)
        
        indices = json.loads(response_text)
        filtered = [state["extracted_jobs"][i] for i in indices if i < len(state["extracted_jobs"])]
        state["filtered_jobs"] = filtered
        print(f"Filtered to {len(filtered)} jobs")
    except Exception as e:
        print(f"Filter error: {e}, keeping all jobs")
        state["filtered_jobs"] = state["extracted_jobs"]
    
    return state

In [10]:
# Cell 10: FIXED - Handle content as list
def rank_jobs(state: JobAgentState) -> JobAgentState:
    """Rank jobs based on user skills or resume"""
    llm = get_llm()
    
    if not state["filtered_jobs"]:
        state["ranked_jobs"] = []
        return state
    
    user_context = ""
    if state.get("user_skills"):
        user_context = f"User skills: {', '.join(state['user_skills'])}"
    elif state.get("user_resume"):
        user_context = f"User resume: {state['user_resume'][:500]}"
    
    if not user_context:
        state["ranked_jobs"] = state["filtered_jobs"]
        return state
    
    prompt = ChatPromptTemplate.from_template(
        """Rank these jobs by how well they match the user profile. Return indices in order from best to worst match.

{user_context}

Jobs: {jobs}

Return JSON list of indices (0-indexed) sorted by match quality.
Example: [3, 0, 5, 1, 2]

Return ONLY the JSON list."""
    )
    
    try:
        chain = prompt | llm
        result = chain.invoke({
            "user_context": user_context,
            "jobs": json.dumps(state["filtered_jobs"][:15], ensure_ascii=False)
        })
        
        # Fix: Extract text from content list
        if hasattr(result, 'content'):
            if isinstance(result.content, list):
                response_text = result.content[0]['text'].strip()
            else:
                response_text = result.content.strip()
        else:
            response_text = str(result).strip()
        
        response_text = re.sub(r'^```json\s*', '', response_text)
        response_text = re.sub(r'\s*```$', '', response_text)
        
        indices = json.loads(response_text)
        ranked = [state["filtered_jobs"][i] for i in indices if i < len(state["filtered_jobs"])]
        state["ranked_jobs"] = ranked
        print(f"Ranked {len(ranked)} jobs")
    except Exception as e:
        print(f"Ranking error: {e}")
        state["ranked_jobs"] = state["filtered_jobs"]
    
    return state

In [11]:
# Cell 11: FIXED - Handle content as list
def generate_answer(state: JobAgentState) -> JobAgentState:
    """Generate natural language summary"""
    llm = get_llm()
    
    if not state["ranked_jobs"]:
        state["final_answer"] = "No jobs found matching your query. Try different keywords or location."
        return state
    
    top_jobs = state["ranked_jobs"][:5]
    
    prompt = ChatPromptTemplate.from_template(
        """Create a brief summary of these job opportunities for the user.

Query: {query}
Jobs: {jobs}

Write 2-3 sentences highlighting:
- How many jobs found
- Top companies or roles
- Key requirements or salary ranges

Keep it simple and direct."""
    )
    
    chain = prompt | llm
    result = chain.invoke({
        "query": state["query"],
        "jobs": json.dumps(top_jobs, ensure_ascii=False)
    })
    
    # Fix: Extract text from content list
    if hasattr(result, 'content'):
        if isinstance(result.content, list):
            state["final_answer"] = result.content[0]['text'].strip()
        else:
            state["final_answer"] = result.content.strip()
    else:
        state["final_answer"] = str(result).strip()
    
    return state

In [12]:
# Cell 12: Node 8 - Save to JSON file
def save_results(state: JobAgentState) -> JobAgentState:
    """Save extracted jobs to JSON file"""
    filename = f"jobs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    
    output = {
        "query": state["query"],
        "timestamp": datetime.now().isoformat(),
        "total_jobs": len(state["ranked_jobs"]),
        "jobs": state["ranked_jobs"]
    }
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    print(f"Saved results to {filename}")
    return state

In [13]:
# Cell 13: Build the agent workflow
def create_job_agent():
    """Create LangGraph workflow"""
    workflow = StateGraph(JobAgentState)
    
    # Add nodes
    workflow.add_node("understand", understand_query)
    workflow.add_node("search", search_web)
    workflow.add_node("scrape", scrape_content)
    workflow.add_node("extract", extract_jobs)
    workflow.add_node("filter", filter_jobs)
    workflow.add_node("rank", rank_jobs)
    workflow.add_node("answer", generate_answer)
    workflow.add_node("save", save_results)
    
    # Define flow
    workflow.set_entry_point("understand")
    workflow.add_edge("understand", "search")
    workflow.add_edge("search", "scrape")
    workflow.add_edge("scrape", "extract")
    workflow.add_edge("extract", "filter")
    workflow.add_edge("filter", "rank")
    workflow.add_edge("rank", "answer")
    workflow.add_edge("answer", "save")
    workflow.add_edge("save", END)
    
    return workflow.compile()

In [14]:
# Cell 14: Main function to run agent
def search_jobs(
    query: str,
    user_skills: List[str] = None,
    user_resume: str = None
):
    """
    Search for jobs and return results
    
    Args:
        query: Natural language job search query
        user_skills: List of user skills for ranking
        user_resume: User resume text for ranking
    
    Returns:
        dict with 'answer' and 'jobs'
    """
    agent = create_job_agent()
    
    initial_state = {
        "query": query,
        "search_keywords": "",
        "search_results": [],
        "scraped_content": [],
        "extracted_jobs": [],
        "filtered_jobs": [],
        "ranked_jobs": [],
        "final_answer": "",
        "user_skills": user_skills or [],
        "user_resume": user_resume or ""
    }
    
    print(f"\nSearching for: {query}\n")
    print("=" * 60)
    
    result = agent.invoke(initial_state)
    
    print("=" * 60)
    print(f"\n{result['final_answer']}\n")
    
    return {
        "answer": result["final_answer"],
        "jobs": result["ranked_jobs"]
    }

In [16]:
# Cell 15: Example 1 - Basic job search
result1 = search_jobs("Accountant jobs Ho Chi Minh City")

print("\nTop 10 Jobs:")
for i, job in enumerate(result1["jobs"][:10], 1):
    print(f"\n{i}. {job.get('title', 'N/A')} at {job.get('company', 'N/A')}")
    print(f"   Location: {job.get('location', 'N/A')}")
    print(f"   Salary: {job.get('salary', 'Not specified')}")
    print(f"   Skills: {', '.join(job.get('skills', [])[:5])}")


Searching for: Accountant jobs Ho Chi Minh City

Search keywords: Accountant jobs Ho Chi Minh City Vietnam careers
Tavily found 10 URLs for: Accountant jobs Ho Chi Minh City Vietnam careers
Scraped: https://www.michaelpage.com.vn/jobs/accountant/ho-...
Scraped: https://jobs.vn.indeed.com/q-accountant-l-th%C3%A0...
Scraped: https://careers.accor.com/global/en/job/accountant...
Scraped: https://www.careerjet.vn/accountant-jobs/Ho-Chi-Mi...
Scraped: https://www.linkedin.com/jobs/search/?currentJobId...
Extracted 4 jobs from https://www.michaelpage.com.vn/jobs/accountant/ho-...
Extracted 0 jobs from https://jobs.vn.indeed.com/q-accountant-l-th%C3%A0...
Extracted 1 jobs from https://careers.accor.com/global/en/job/accountant...
Extracted 0 jobs from https://www.careerjet.vn/accountant-jobs/Ho-Chi-Mi...
Extracted 17 jobs from https://www.linkedin.com/jobs/search/?currentJobId...
Total extracted jobs: 22
Filtered to 17 jobs
Saved results to jobs_20260512_231038.json

There are 5 accountant j